# GTE multilingual reranker ablation (development only)

이 노트북은 **Codex coder agent**가 fresh kernel에서 실행한 개발셋 전용 실험이다. 독립 holdout은 사용하지 않으며 운영·미관측 데이터 일반화를 주장하지 않는다.

사전 고정 계약: 저장된 16번 RRF 후보만 사용하고 HNSW를 재질의하지 않는다. `0.5:0.5` 및 `0.4:0.6` 각각 Top20/Top50을 GTE logit만으로 재정렬한다. 입력은 원본 query와 `chunk.document`뿐이다. `max_length=8192`, `truncation='only_second'`, query 보존, logit 동점은 기존 RRF rank 순이다. primary는 evidence MRR@5이며 공식 no-reranker 기준 대비 `+0.025` 이상과 모든 guardrail 비회귀를 동시에 만족해야 승격한다.


In [1]:
import os
os.environ['HF_HOME'] = str((__import__('pathlib').Path.cwd().parents[0] / '.cache/huggingface').resolve())
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert os.environ.get('CUDA_VISIBLE_DEVICES') == '0'

import csv, gc, hashlib, importlib.metadata, json, math, statistics, tempfile, time
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

PROJECT_ROOT = Path.cwd().parents[0].resolve()
OUTPUT_ROOT = PROJECT_ROOT / 'notebooks/data/17_gte_reranker_ablation'
SOURCE_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
WEIGHT_ROOT = PROJECT_ROOT / 'notebooks/data/16_normalized_rrf_weight_ablation'
MODEL_ROOT = PROJECT_ROOT / '.cache/reranker/gte-multilingual-reranker-base'
CUSTOM_HUB_ROOT = PROJECT_ROOT / '.cache/huggingface/hub/models--Alibaba-NLP--new-impl/snapshots/40ced75c3017eb27626c9d4ea981bde21a2662f4'
CUSTOM_MODULE_ROOT = PROJECT_ROOT / '.cache/huggingface/modules/transformers_modules/Alibaba_hyphen_NLP/new_hyphen_impl/40ced75c3017eb27626c9d4ea981bde21a2662f4'
MODEL_REVISION = '8215cf04918ba6f7b6a62bb44238ce2953d8831c'
CUSTOM_CODE_REVISION = '40ced75c3017eb27626c9d4ea981bde21a2662f4'
MAX_LENGTH, BATCH_SIZE, PRIMARY_DELTA, TOLERANCE = 8192, 2, 0.025, 1e-12
WEIGHTS = {'vector_0.5_bm25_0.5': (0.5, 0.5), 'vector_0.4_bm25_0.6': (0.4, 0.6)}
TOP_KS = (20, 50)
RERANK_CONFIGS = {f'{weight}_top{topk}': {'weight': weight, 'top_k': topk} for weight in WEIGHTS for topk in TOP_KS}
BASELINE_CONFIGS = {weight: f'{weight}_no_reranker' for weight in WEIGHTS}
OFFICIAL_BASELINE = 'vector_0.5_bm25_0.5_no_reranker'
METRICS = ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5')
SELECTION_RULE = {'official_baseline': OFFICIAL_BASELINE, 'primary': 'evidence_mrr_at_5', 'minimum_delta': PRIMARY_DELTA, 'guardrails': ['evidence_strict_evidence_hit_at_3', 'evidence_recall_at_5', 'evidence_ndcg_at_5', 'evidence_card_hit_at_3', 'card_card_hit_at_3'], 'winner_order': ['mrr_at_5', 'ndcg_at_5', 'recall_at_5', 'strict_evidence_hit_at_3', 'lower_top_k']}
assert len(RERANK_CONFIGS) == 4 and torch.cuda.is_available() and torch.cuda.device_count() == 1
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'contract_frozen_before_results': True, 'physical_gpu': 0, 'visible_device': 0, 'gpu': torch.cuda.get_device_name(0), 'configs': RERANK_CONFIGS, 'selection': SELECTION_RULE})


/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'contract_frozen_before_results': True, 'physical_gpu': 0, 'visible_device': 0, 'gpu': 'NVIDIA GeForce RTX 3090', 'configs': {'vector_0.5_bm25_0.5_top20': {'weight': 'vector_0.5_bm25_0.5', 'top_k': 20}, 'vector_0.5_bm25_0.5_top50': {'weight': 'vector_0.5_bm25_0.5', 'top_k': 50}, 'vector_0.4_bm25_0.6_top20': {'weight': 'vector_0.4_bm25_0.6', 'top_k': 20}, 'vector_0.4_bm25_0.6_top50': {'weight': 'vector_0.4_bm25_0.6', 'top_k': 50}}, 'selection': {'official_baseline': 'vector_0.5_bm25_0.5_no_reranker', 'primary': 'evidence_mrr_at_5', 'minimum_delta': 0.025, 'guardrails': ['evidence_strict_evidence_hit_at_3', 'evidence_recall_at_5', 'evidence_ndcg_at_5', 'evidence_card_hit_at_3', 'card_card_hit_at_3'], 'winner_order': ['mrr_at_5', 'ndcg_at_5', 'recall_at_5', 'strict_evidence_hit_at_3', 'lower_top_k']}}


In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def files_under(root):
    paths = []
    for directory, subdirectories, names in os.walk(root):
        subdirectories.sort()
        paths.extend(Path(directory) / name for name in sorted(names))
    return paths

def tree_snapshot(root):
    files = {path.relative_to(root).as_posix(): {'sha256': sha256_file(path), 'bytes': path.stat().st_size} for path in files_under(root)}
    digest = hashlib.sha256()
    for relative, item in files.items():
        digest.update(relative.encode())
        digest.update(bytes.fromhex(item['sha256']))
    return {'file_count': len(files), 'total_bytes': sum(item['bytes'] for item in files.values()), 'tree_sha256': digest.hexdigest(), 'files': files}

def write_csv(path, rows):
    rows = list(rows)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader(); writer.writerows(rows)
    os.replace(temporary, path)

def write_json(path, value):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        json.dump(value, handle, ensure_ascii=False, indent=2); handle.write('\n')
    os.replace(temporary, path)

def normalized_text(value):
    import unicodedata
    return ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())

SOURCE_FILES = {
    'chunks': SOURCE_ROOT / 'chunks.jsonl',
    'retrieval_per_query': SOURCE_ROOT / 'retrieval_per_query.csv',
    'weight_candidates': WEIGHT_ROOT / 'rrf_weight_candidates.csv',
    'weight_per_query': WEIGHT_ROOT / 'rrf_weight_ablation_per_query.csv',
    'weight_summary': WEIGHT_ROOT / 'rrf_weight_ablation_summary.json',
    'weight_manifest': WEIGHT_ROOT / 'rrf_weight_run_manifest.json',
}
source_hashes_before = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}
cache_before = {'model': tree_snapshot(MODEL_ROOT), 'custom_hub': tree_snapshot(CUSTOM_HUB_ROOT), 'custom_modules': tree_snapshot(CUSTOM_MODULE_ROOT)}
assert (MODEL_ROOT / '.cache/huggingface/download/config.json.metadata').read_text().splitlines()[0] == MODEL_REVISION

chunks = [json.loads(line) for line in SOURCE_FILES['chunks'].read_text(encoding='utf-8').splitlines()]
chunk_by_id = {chunk['id']: chunk for chunk in chunks}
base_rows = list(csv.DictReader(SOURCE_FILES['retrieval_per_query'].open(encoding='utf-8', newline='')))
evaluations = {}
for row in base_rows:
    if row['method'] == 'keyword':
        evaluations[row['query_id']] = {'query_id': row['query_id'], 'query': row['query'], 'category': row['category'], 'expected_card': row['expected_card'], 'expected_level': row['expected_level'], 'required_terms': json.loads(row['required_terms'])}
assert len(chunks) == 327 and len(evaluations) == 30
assert sum(item['expected_level'] == 'card' for item in evaluations.values()) == 10
candidate_rows_16 = list(csv.DictReader(SOURCE_FILES['weight_candidates'].open(encoding='utf-8', newline='')))
candidate_rows_16 = [row for row in candidate_rows_16 if row['configuration'] in WEIGHTS]
assert len(candidate_rows_16) == 2 * 30 * 50
rankings = {}
for weight in WEIGHTS:
    for query_id in evaluations:
        selected = sorted((row for row in candidate_rows_16 if row['configuration'] == weight and row['query_id'] == query_id), key=lambda row: int(row['fused_rank']))
        assert [int(row['fused_rank']) for row in selected] == list(range(1, 51))
        rankings[(weight, query_id)] = [row['chunk_id'] for row in selected]
        assert rankings[(weight, query_id)][:20] == rankings[(weight, query_id)][:50][:20]
        assert len(set(rankings[(weight, query_id)])) == 50

def relevant_ids(evaluation):
    return {chunk['id'] for chunk in chunks if chunk['metadata']['card_key'] == evaluation['expected_card'] and chunk['metadata']['level'] == evaluation['expected_level'] and all(normalized_text(term) in normalized_text(chunk['document']) for term in evaluation['required_terms'])}

def calculate_metrics(evaluation, ranking):
    relevant = relevant_ids(evaluation)
    hits = [identifier in relevant for identifier in ranking[:5]]
    first = next((rank for rank, hit in enumerate(hits, 1) if hit), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1))
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return {'card_hit_at_3': int(any(chunk_by_id[i]['metadata']['card_key'] == evaluation['expected_card'] for i in ranking[:3])), 'strict_evidence_hit_at_3': int(any(hits[:3])), 'recall_at_5': sum(hits) / len(relevant), 'mrr_at_5': 1 / first if first else 0.0, 'ndcg_at_5': dcg / ideal if ideal else 0.0}

saved_16 = list(csv.DictReader(SOURCE_FILES['weight_per_query'].open(encoding='utf-8', newline='')))
saved_16 = {(row['configuration'], row['query_id']): row for row in saved_16 if row['configuration'] in WEIGHTS}
for weight in WEIGHTS:
    for query_id, evaluation in evaluations.items():
        saved = saved_16[(weight, query_id)]; ranking = rankings[(weight, query_id)]
        assert ranking[:5] == json.loads(saved['top5_chunk_ids'])
        current = calculate_metrics(evaluation, ranking)
        assert all(math.isclose(current[name], float(saved[name]), rel_tol=0, abs_tol=TOLERANCE) for name in METRICS)

pair_keys = sorted({(query_id, chunk_id) for weight in WEIGHTS for query_id in evaluations for chunk_id in rankings[(weight, query_id)]})
assert len(pair_keys) == 1857
pair_membership = defaultdict(list)
for config, spec in RERANK_CONFIGS.items():
    for query_id in evaluations:
        for chunk_id in rankings[(spec['weight'], query_id)][:spec['top_k']]: pair_membership[(query_id, chunk_id)].append(config)
assert set(pair_membership) == set(pair_keys)
print({'chunks': len(chunks), 'queries': len(evaluations), 'candidate_rows': len(candidate_rows_16), 'unique_pairs': len(pair_keys), 'baseline_top5_and_metrics_exact': True, 'top20_prefix_assertions': 60, 'model_cache': {k: {'files': v['file_count'], 'bytes': v['total_bytes'], 'sha256': v['tree_sha256']} for k, v in cache_before.items()}})


{'chunks': 327, 'queries': 30, 'candidate_rows': 3000, 'unique_pairs': 1857, 'baseline_top5_and_metrics_exact': True, 'top20_prefix_assertions': 60, 'model_cache': {'model': {'files': 27, 'bytes': 629269531, 'sha256': 'b9d43d81d88cf37b950f6208b48172e1a7c98fdca2d7b0b7a878eb423e2b1f49'}, 'custom_hub': {'files': 2, 'bytes': 66150, 'sha256': '490abe8313b2b20dac3178d76b3e18f9f0f494888f944f325ee9ce723b8e19d8'}, 'custom_modules': {'files': 5, 'bytes': 141830, 'sha256': '0265f955cf07d667a488656cbc4a0279bbf48a2ef6982cea495943ad8060665d'}}}


## Local-only model load, token audit, and single-pass scoring

품질 pair는 합집합 기준으로 정확히 한 번만 점수화한다. 짧은 microbenchmark의 실제 품질 점수도 동일 cache에 넣고 본 실행에서 재계산하지 않는다. OOM이면 batch·길이 정책을 바꾸지 않고 실패시킨다.


In [3]:
versions = {name: importlib.metadata.version(name) for name in ('transformers', 'tokenizers', 'huggingface-hub', 'torch', 'safetensors', 'sentencepiece', 'einops')}
assert versions == {'transformers': '4.57.6', 'tokenizers': '0.22.2', 'huggingface-hub': '0.36.2', 'torch': '2.11.0', 'safetensors': '0.8.0', 'sentencepiece': '0.2.2', 'einops': '0.8.2'}
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)
load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ROOT, local_files_only=True)
assert tokenizer.model_max_length == 32768 and MAX_LENGTH <= tokenizer.model_max_length
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ROOT, trust_remote_code=True, local_files_only=True, code_revision=CUSTOM_CODE_REVISION, dtype=torch.float16)
model.eval().to('cuda:0'); torch.cuda.synchronize()
model_load_seconds = time.perf_counter() - load_started
assert next(model.parameters()).dtype == torch.float16 and next(model.parameters()).device.type == 'cuda'

raw_query_tokens = {query_id: len(tokenizer(evaluation['query'], add_special_tokens=False)['input_ids']) for query_id, evaluation in evaluations.items()}
unique_chunks = sorted({chunk_id for _, chunk_id in pair_keys})
raw_document_tokens = {chunk_id: len(tokenizer(chunk_by_id[chunk_id]['document'], add_special_tokens=False)['input_ids']) for chunk_id in unique_chunks}
special_pair_tokens = tokenizer.num_special_tokens_to_add(pair=True)
assert special_pair_tokens == 4 and all(count + special_pair_tokens < MAX_LENGTH for count in raw_query_tokens.values())
pair_lengths = {(query_id, chunk_id): raw_query_tokens[query_id] + raw_document_tokens[chunk_id] + special_pair_tokens for query_id, chunk_id in pair_keys}
ordered_pairs = sorted(pair_keys, key=lambda pair: (pair_lengths[pair], pair))
score_cache, token_rows = {}, []

def score_pairs(batch_pairs):
    queries = [evaluations[q]['query'] for q, _ in batch_pairs]
    documents = [chunk_by_id[c]['document'] for _, c in batch_pairs]
    encoded = tokenizer(queries, documents, padding=True, truncation='only_second', max_length=MAX_LENGTH, return_tensors='pt')
    input_counts = encoded['attention_mask'].sum(dim=1).tolist()
    device_inputs = {key: value.to('cuda:0', non_blocking=True) for key, value in encoded.items()}
    with torch.inference_mode(): logits = model(**device_inputs).logits.reshape(-1).float().cpu().tolist()
    assert len(logits) == len(batch_pairs) and all(math.isfinite(score) for score in logits)
    for pair, score, input_count in zip(batch_pairs, logits, input_counts):
        query_id, chunk_id = pair; input_document = input_count - raw_query_tokens[query_id] - special_pair_tokens
        truncated = raw_document_tokens[chunk_id] - input_document
        assert pair not in score_cache and 0 <= input_document <= raw_document_tokens[chunk_id] and truncated >= 0 and input_count <= MAX_LENGTH
        score_cache[pair] = score
        token_rows.append({'query_id': query_id, 'chunk_id': chunk_id, 'reranker_logit': score, 'original_rrf_ranks': json.dumps({config: rankings[(spec['weight'], query_id)].index(chunk_id) + 1 for config, spec in RERANK_CONFIGS.items() if chunk_id in rankings[(spec['weight'], query_id)][:spec['top_k']]}, sort_keys=True), 'configurations': json.dumps(sorted(pair_membership[pair])), 'level': chunk_by_id[chunk_id]['metadata']['level'], 'query_tokens_raw': raw_query_tokens[query_id], 'document_tokens_raw': raw_document_tokens[chunk_id], 'input_tokens': input_count, 'document_tokens_input': input_document, 'document_tokens_truncated': truncated, 'document_truncation_ratio': truncated / raw_document_tokens[chunk_id] if raw_document_tokens[chunk_id] else 0.0})

warmup_started = time.perf_counter()
warm = tokenizer(['워밍업 질문'], ['짧은 문서'], padding=True, truncation='only_second', max_length=MAX_LENGTH, return_tensors='pt')
with torch.inference_mode(): _ = model(**{key: value.to('cuda:0') for key, value in warm.items()}).logits
torch.cuda.synchronize(); warmup_seconds = time.perf_counter() - warmup_started
quality_started = time.perf_counter()
micro_pairs = ordered_pairs[:BATCH_SIZE]
micro_started = time.perf_counter(); score_pairs(micro_pairs); torch.cuda.synchronize(); microbenchmark_seconds = time.perf_counter() - micro_started
remaining = ordered_pairs[BATCH_SIZE:]
try:
    for start in range(0, len(remaining), BATCH_SIZE): score_pairs(remaining[start:start + BATCH_SIZE])
    torch.cuda.synchronize()
except torch.cuda.OutOfMemoryError:
    raise RuntimeError('OOM under predeclared batch_size=2 and max_length=8192; policy was not changed')
quality_scoring_seconds = time.perf_counter() - quality_started
assert set(score_cache) == set(pair_keys) and len(token_rows) == len(pair_keys)
peak_allocated = torch.cuda.max_memory_allocated(0); peak_reserved = torch.cuda.max_memory_reserved(0)
print({'model_load_seconds': model_load_seconds, 'warmup_seconds': warmup_seconds, 'microbenchmark_pairs': len(micro_pairs), 'microbenchmark_seconds': microbenchmark_seconds, 'quality_pairs': len(score_cache), 'quality_scoring_seconds': quality_scoring_seconds, 'pairs_per_second': len(score_cache) / quality_scoring_seconds, 'peak_allocated_bytes': peak_allocated, 'peak_reserved_bytes': peak_reserved, 'truncated_pairs': sum(row['document_tokens_truncated'] > 0 for row in token_rows)})


{'model_load_seconds': 0.9209258519113064, 'warmup_seconds': 0.2606452889740467, 'microbenchmark_pairs': 2, 'microbenchmark_seconds': 0.029173234943300486, 'quality_pairs': 1857, 'quality_scoring_seconds': 9.171242276905105, 'pairs_per_second': 202.48074840158478, 'peak_allocated_bytes': 794649600, 'peak_reserved_bytes': 1283457024, 'truncated_pairs': 0}


In [4]:
reranked = {}
for config, spec in RERANK_CONFIGS.items():
    for query_id in evaluations:
        pool = rankings[(spec['weight'], query_id)][:spec['top_k']]
        old_rank = {chunk_id: rank for rank, chunk_id in enumerate(pool, 1)}
        result = sorted(pool, key=lambda chunk_id: (-score_cache[(query_id, chunk_id)], old_rank[chunk_id]))
        assert len(result) == len(pool) and set(result) == set(pool)
        reranked[(config, query_id)] = result

all_rankings = {}
for weight, baseline_config in BASELINE_CONFIGS.items():
    for query_id in evaluations: all_rankings[(baseline_config, query_id)] = rankings[(weight, query_id)]
all_rankings.update(reranked)

per_query_rows = []
baseline_metric_lookup = {}
for weight, baseline_config in BASELINE_CONFIGS.items():
    for query_id, evaluation in evaluations.items(): baseline_metric_lookup[(weight, query_id)] = calculate_metrics(evaluation, rankings[(weight, query_id)])
for (config, query_id), ranking in all_rankings.items():
    evaluation = evaluations[query_id]; current = calculate_metrics(evaluation, ranking)
    weight = config.removesuffix('_no_reranker') if config.endswith('_no_reranker') else RERANK_CONFIGS[config]['weight']
    baseline = baseline_metric_lookup[(weight, query_id)]
    per_query_rows.append({'configuration': config, 'system': 'no_reranker' if config.endswith('_no_reranker') else 'gte_reranker', 'candidate_weight': weight, 'top_k': 50 if config.endswith('_no_reranker') else RERANK_CONFIGS[config]['top_k'], 'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence', 'category': evaluation['category'], **current, **{f'delta_{name}': current[name] - baseline[name] for name in METRICS}, 'top5_chunk_ids': json.dumps(ranking[:5], separators=(',', ':')), 'top5_cards': json.dumps([chunk_by_id[i]['metadata']['card_key'] for i in ranking[:5]], ensure_ascii=False, separators=(',', ':')), 'top5_levels': json.dumps([chunk_by_id[i]['metadata']['level'] for i in ranking[:5]], separators=(',', ':'))})
assert all(math.isfinite(float(row[name])) and 0 <= float(row[name]) <= 1 for row in per_query_rows for name in METRICS)

groups = [('all', lambda row: True), ('card', lambda row: row['question_group'] == 'card'), ('evidence', lambda row: row['question_group'] == 'evidence')]
for category in sorted({evaluation['category'] for evaluation in evaluations.values()}): groups.append((f'category:{category}', lambda row, category=category: row['category'] == category))
summary_rows = []
for config in [*BASELINE_CONFIGS.values(), *RERANK_CONFIGS]:
    config_rows = [row for row in per_query_rows if row['configuration'] == config]
    for group, predicate in groups:
        selected = [row for row in config_rows if predicate(row)]
        summary_rows.append({'configuration': config, 'system': selected[0]['system'], 'candidate_weight': selected[0]['candidate_weight'], 'top_k': selected[0]['top_k'], 'question_group': group, 'denominator': len(selected), **{name: sum(float(row[name]) for row in selected) / len(selected) for name in METRICS}})
summary_map = {(row['configuration'], row['question_group']): row for row in summary_rows}

ceiling_rows = []
for config, spec in RERANK_CONFIGS.items():
    for query_id, evaluation in evaluations.items():
        candidates = rankings[(spec['weight'], query_id)][:spec['top_k']]; relevant = relevant_ids(evaluation); present = sorted(relevant.intersection(candidates))
        ceiling_rows.append({'configuration': config, 'candidate_weight': spec['weight'], 'top_k': spec['top_k'], 'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence', 'relevant_count': len(relevant), 'candidate_count': len(candidates), 'strict_candidate_hit': int(bool(present)), 'strict_candidate_recall': len(present) / len(relevant), 'expected_card_candidate_hit': int(any(chunk_by_id[i]['metadata']['card_key'] == evaluation['expected_card'] for i in candidates)), 'missing_relevant_ids': json.dumps(sorted(relevant.difference(candidates)), separators=(',', ':'))})

official_evidence = summary_map[(OFFICIAL_BASELINE, 'evidence')]; official_card = summary_map[(OFFICIAL_BASELINE, 'card')]
qualification = {}
for config, spec in RERANK_CONFIGS.items():
    evidence, card = summary_map[(config, 'evidence')], summary_map[(config, 'card')]; delta = evidence['mrr_at_5'] - official_evidence['mrr_at_5']
    guardrails = {'evidence_strict_evidence_hit_at_3': evidence['strict_evidence_hit_at_3'] >= official_evidence['strict_evidence_hit_at_3'] - TOLERANCE, 'evidence_recall_at_5': evidence['recall_at_5'] >= official_evidence['recall_at_5'] - TOLERANCE, 'evidence_ndcg_at_5': evidence['ndcg_at_5'] >= official_evidence['ndcg_at_5'] - TOLERANCE, 'evidence_card_hit_at_3': evidence['card_hit_at_3'] >= official_evidence['card_hit_at_3'] - TOLERANCE, 'card_card_hit_at_3': card['card_hit_at_3'] >= official_card['card_hit_at_3'] - TOLERANCE}
    qualification[config] = {'evidence_mrr_delta': delta, 'primary_pass': delta >= PRIMARY_DELTA - TOLERANCE, 'guardrails': guardrails, 'qualified': delta >= PRIMARY_DELTA - TOLERANCE and all(guardrails.values())}
qualified = [config for config in RERANK_CONFIGS if qualification[config]['qualified']]
winner_key = lambda config: (-summary_map[(config, 'evidence')]['mrr_at_5'], -summary_map[(config, 'evidence')]['ndcg_at_5'], -summary_map[(config, 'evidence')]['recall_at_5'], -summary_map[(config, 'evidence')]['strict_evidence_hit_at_3'], RERANK_CONFIGS[config]['top_k'], config)
selected = min(qualified, key=winner_key) if qualified else OFFICIAL_BASELINE
raw_best = min(RERANK_CONFIGS, key=winner_key)
paired = {}
official_evidence_rows = {row['query_id']: row for row in per_query_rows if row['configuration'] == OFFICIAL_BASELINE and row['question_group'] == 'evidence'}
for config in RERANK_CONFIGS:
    paired[config] = {}
    for metric in ('mrr_at_5', 'ndcg_at_5'):
        deltas = [float(row[metric]) - float(official_evidence_rows[row['query_id']][metric]) for row in per_query_rows if row['configuration'] == config and row['question_group'] == 'evidence']
        paired[config][metric] = {'wins': sum(delta > TOLERANCE for delta in deltas), 'losses': sum(delta < -TOLERANCE for delta in deltas), 'ties': sum(abs(delta) <= TOLERANCE for delta in deltas), 'mean_delta': sum(deltas) / len(deltas)}
print({'official_baseline_evidence': official_evidence, 'raw_best': raw_best, 'qualification': qualification, 'selected': selected})


{'official_baseline_evidence': {'configuration': 'vector_0.5_bm25_0.5_no_reranker', 'system': 'no_reranker', 'candidate_weight': 'vector_0.5_bm25_0.5', 'top_k': 50, 'question_group': 'evidence', 'denominator': 20, 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.75, 'recall_at_5': 0.7125, 'mrr_at_5': 0.6033333333333333, 'ndcg_at_5': 0.5965505415086586}, 'raw_best': 'vector_0.4_bm25_0.6_top50', 'qualification': {'vector_0.5_bm25_0.5_top20': {'evidence_mrr_delta': -0.10999999999999993, 'primary_pass': False, 'guardrails': {'evidence_strict_evidence_hit_at_3': False, 'evidence_recall_at_5': False, 'evidence_ndcg_at_5': False, 'evidence_card_hit_at_3': True, 'card_card_hit_at_3': True}, 'qualified': False}, 'vector_0.5_bm25_0.5_top50': {'evidence_mrr_delta': -0.10250000000000004, 'primary_pass': False, 'guardrails': {'evidence_strict_evidence_hit_at_3': False, 'evidence_recall_at_5': False, 'evidence_ndcg_at_5': False, 'evidence_card_hit_at_3': True, 'card_card_hit_at_3': True}, 'qualif

In [5]:
def percentile(values, q): return float(np.percentile(np.asarray(values, dtype=float), q))
def truncation_stats(rows):
    return {'pairs': len(rows), 'truncated_pairs': sum(int(row['document_tokens_truncated']) > 0 for row in rows), 'truncated_pair_rate': sum(int(row['document_tokens_truncated']) > 0 for row in rows) / len(rows), 'document_truncation_ratio_median': statistics.median(float(row['document_truncation_ratio']) for row in rows), 'document_truncation_ratio_p95': percentile([row['document_truncation_ratio'] for row in rows], 95), 'input_tokens_median': statistics.median(int(row['input_tokens']) for row in rows), 'input_tokens_p95': percentile([row['input_tokens'] for row in rows], 95), 'input_tokens_max': max(int(row['input_tokens']) for row in rows)}
token_stats_by_level = {level: truncation_stats([row for row in token_rows if row['level'] == level]) for level in sorted({row['level'] for row in token_rows})}
token_stats_by_configuration = {}
for config in RERANK_CONFIGS:
    rows = [row for row in token_rows if config in json.loads(row['configurations'])]
    token_stats_by_configuration[config] = truncation_stats(rows)

resource_result = {'physical_gpu_index': 0, 'cuda_visible_devices': os.environ['CUDA_VISIBLE_DEVICES'], 'visible_cuda_index': 0, 'gpu_name': torch.cuda.get_device_name(0), 'dtype': 'float16', 'batch_size': BATCH_SIZE, 'max_length': MAX_LENGTH, 'truncation': 'only_second', 'model_load_seconds': model_load_seconds, 'warmup_seconds': warmup_seconds, 'microbenchmark': {'pairs': len(micro_pairs), 'seconds': microbenchmark_seconds, 'pairs_per_second': len(micro_pairs) / microbenchmark_seconds}, 'quality_scoring': {'unique_pairs': len(pair_keys), 'seconds_measured': quality_scoring_seconds, 'pairs_per_second': len(pair_keys) / quality_scoring_seconds}, 'estimated_per_configuration_latency': {config: {'seconds_estimated': (30 * spec['top_k']) / (len(pair_keys) / quality_scoring_seconds), 'method': 'candidate_pairs / measured unique-pair throughput; not separately rerun'} for config, spec in RERANK_CONFIGS.items()}, 'peak_vram_allocated_bytes': peak_allocated, 'peak_vram_reserved_bytes': peak_reserved, 'network_calls': 0, 'api_calls': 0, 'new_embeddings': 0, 'package_installs': 0}
write_csv(OUTPUT_ROOT / 'gte_reranker_per_query.csv', per_query_rows)
write_csv(OUTPUT_ROOT / 'gte_reranker_summary.csv', summary_rows)
write_csv(OUTPUT_ROOT / 'gte_reranker_pair_scores.csv', sorted(token_rows, key=lambda row: (row['query_id'], row['chunk_id'])))
write_csv(OUTPUT_ROOT / 'gte_reranker_candidate_ceiling.csv', ceiling_rows)
write_json(OUTPUT_ROOT / 'gte_reranker_resources.json', resource_result)

source_hashes_after = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}
cache_after = {'model': tree_snapshot(MODEL_ROOT), 'custom_hub': tree_snapshot(CUSTOM_HUB_ROOT), 'custom_modules': tree_snapshot(CUSTOM_MODULE_ROOT)}
assert source_hashes_after == source_hashes_before and cache_after == cache_before
summary_json = {'schema_version': 'gte_reranker_ablation_v1', 'scope': {'development_queries': 30, 'card_queries': 10, 'evidence_queries': 20, 'chunks': 327, 'holdout_used': False, 'claim': 'development candidate comparison only'}, 'contract': {'model_revision': MODEL_REVISION, 'custom_code_revision': CUSTOM_CODE_REVISION, 'configurations': RERANK_CONFIGS, 'official_no_reranker': OFFICIAL_BASELINE, 'selection_rule_frozen_before_results': SELECTION_RULE, 'model_input_fields': ['original_query', 'chunk.document'], 'ranking': 'reranker logit descending; ties by original RRF rank; no score mixing', 'max_length': MAX_LENGTH, 'truncation': 'only_second', 'candidate_source': 'stored 16 candidate CSV; no HNSW query'}, 'versions': versions, 'baseline_reproduction': {'weights': list(WEIGHTS), 'top5_ids_and_metrics_exact_queries_each': 30}, 'results': {'summaries': summary_rows, 'paired_evidence_vs_official_baseline': paired, 'candidate_ceiling': {'rows': len(ceiling_rows)}, 'raw_metric_best_reranker_configuration': raw_best}, 'selection': {'qualified_configurations': qualified, 'qualification': qualification, 'selected_configuration': selected, 'decision': 'promote_reranker' if qualified else 'retain_no_reranker', 'raw_metric_best_configuration': raw_best}, 'token_truncation': {'unique_pairs': truncation_stats(token_rows), 'by_level': token_stats_by_level, 'by_configuration': token_stats_by_configuration}, 'resources': resource_result, 'integrity': {'source_hashes_before': source_hashes_before, 'source_hashes_after': source_hashes_after, 'cache_before': cache_before, 'cache_after': cache_after, 'source_and_cache_unchanged': True, 'unique_pair_scored_once': True, 'permutation_assertions': 120, 'top20_prefix_assertions': 60, 'finite_and_range_assertions': True, 'no_leak_assertion': {'model_input_fields_exact': ['original_query', 'chunk.document'], 'gold_or_metadata_fields_used_for_scoring': []}}, 'execution': {'environment': 'skn25', 'fresh_kernel': True, 'cuda_visible_devices': '0', 'physical_gpu_index': 0, 'network_calls': 0, 'api_calls': 0, 'new_embeddings': 0, 'package_installs': 0}, 'korean_glossary': {'Card Hit@3': '상위 3개에 기대 카드가 하나라도 있는 비율', 'Evidence Strict Hit@3': '상위 3개에 엄격 관련 근거 청크가 하나라도 있는 비율', 'Recall@5': '관련 근거 전체 중 상위 5개가 회수한 비율', 'MRR@5': '상위 5개에서 첫 관련 근거 순위의 역수 평균', 'nDCG@5': '상위 5개의 관련 근거 순위 품질', 'candidate ceiling': 'reranker가 순서만 바꿀 때 후보 집합이 허용하는 최대 회수 범위', 'win/loss/tie': '질의별 baseline 대비 개선/하락/동률'}, 'limitations': ['개발 질의 30개만 사용했으며 holdout·운영 일반화 결론이 아니다.', '조합별 latency는 같은 점수 cache를 재사용했기 때문에 별도 실행값이 아니라 전체 처리량 기반 추정치다.', 'reranker는 후보 밖 관련 청크를 새로 회수할 수 없다.']}
write_json(OUTPUT_ROOT / 'gte_reranker_summary.json', summary_json)

readme = f'''# GTE reranker ablation

개발 질의 30개에서 저장된 RRF 후보를 GTE logit으로만 재정렬한 결과다. holdout은 사용하지 않았고 운영 일반화를 주장하지 않는다.

## 지표 풀이

- Card Hit@3: 상위 3개에 기대 카드가 있는 비율
- Evidence Strict Hit@3: 상위 3개에 엄격 관련 근거가 있는 비율
- Recall@5: 관련 근거 중 상위 5개가 찾은 비율
- MRR@5: 첫 관련 근거가 앞에 있을수록 높은 값
- nDCG@5: 관련 근거의 상위 순위 배치 품질
- candidate ceiling: 현재 후보 집합 안에서 가능한 관련 근거 회수 상한
- win/loss/tie: 질의별 baseline 대비 개선/하락/동률

## 재현 계약

- 모델 `{MODEL_REVISION}`, custom code `{CUSTOM_CODE_REVISION}`의 로컬 cache만 사용
- `CUDA_VISIBLE_DEVICES=0`, physical GPU 0, dtype float16, batch {BATCH_SIZE}
- 원본 query + chunk.document만 입력, max_length {MAX_LENGTH}, only_second truncation
- network/API/new embedding/package install 0, HNSW query 0
- 품질용 unique pair {len(pair_keys)}개를 한 번씩만 점수화

선택: `{summary_json['selection']['decision']}` / `{selected}`. Raw metric 최고 reranker 조합은 `{raw_best}`다.
'''
(OUTPUT_ROOT / 'README.md').write_text(readme, encoding='utf-8')

output_names = ['gte_reranker_per_query.csv', 'gte_reranker_summary.csv', 'gte_reranker_summary.json', 'gte_reranker_pair_scores.csv', 'gte_reranker_candidate_ceiling.csv', 'gte_reranker_resources.json', 'README.md']
manifest_files = {f'input:{name}': {'path': str(path.relative_to(PROJECT_ROOT)), 'sha256': source_hashes_before[name], 'bytes': path.stat().st_size} for name, path in SOURCE_FILES.items()}
for cache_name, cache_root in [('model', MODEL_ROOT), ('custom_hub', CUSTOM_HUB_ROOT), ('custom_modules', CUSTOM_MODULE_ROOT)]:
    for relative, item in cache_before[cache_name]['files'].items(): manifest_files[f'input:{cache_name}:{relative}'] = {'path': str((cache_root / relative).relative_to(PROJECT_ROOT)), **item}
for name in output_names:
    path = OUTPUT_ROOT / name; manifest_files[f'output:{name}'] = {'path': str(path.relative_to(PROJECT_ROOT)), 'sha256': sha256_file(path), 'bytes': path.stat().st_size}
manifest_files['output:notebook'] = {'path': 'notebooks/17_gte_reranker_ablation.ipynb', 'sha256': 'pending_after_nbclient_serialization'}
manifest = {'schema_version': 'gte_reranker_run_manifest_v1', 'self_hash_excluded': True, 'model_revision': MODEL_REVISION, 'custom_code_revision': CUSTOM_CODE_REVISION, 'files': manifest_files, 'source_and_cache_unchanged': True}
write_json(OUTPUT_ROOT / 'gte_reranker_run_manifest.json', manifest)
print({'output_rows': {'per_query': len(per_query_rows), 'summary': len(summary_rows), 'pair_scores': len(token_rows), 'candidate_ceiling': len(ceiling_rows)}, 'selection': summary_json['selection'], 'source_cache_unchanged': True})


{'output_rows': {'per_query': 180, 'summary': 36, 'pair_scores': 1857, 'candidate_ceiling': 120}, 'selection': {'qualified_configurations': [], 'qualification': {'vector_0.5_bm25_0.5_top20': {'evidence_mrr_delta': -0.10999999999999993, 'primary_pass': False, 'guardrails': {'evidence_strict_evidence_hit_at_3': False, 'evidence_recall_at_5': False, 'evidence_ndcg_at_5': False, 'evidence_card_hit_at_3': True, 'card_card_hit_at_3': True}, 'qualified': False}, 'vector_0.5_bm25_0.5_top50': {'evidence_mrr_delta': -0.10250000000000004, 'primary_pass': False, 'guardrails': {'evidence_strict_evidence_hit_at_3': False, 'evidence_recall_at_5': False, 'evidence_ndcg_at_5': False, 'evidence_card_hit_at_3': True, 'card_card_hit_at_3': True}, 'qualified': False}, 'vector_0.4_bm25_0.6_top20': {'evidence_mrr_delta': -0.10333333333333328, 'primary_pass': False, 'guardrails': {'evidence_strict_evidence_hit_at_3': False, 'evidence_recall_at_5': False, 'evidence_ndcg_at_5': False, 'evidence_card_hit_at_3': 

In [6]:
stored_per_query = list(csv.DictReader((OUTPUT_ROOT / 'gte_reranker_per_query.csv').open(encoding='utf-8', newline='')))
stored_summary = list(csv.DictReader((OUTPUT_ROOT / 'gte_reranker_summary.csv').open(encoding='utf-8', newline='')))
stored_pairs = list(csv.DictReader((OUTPUT_ROOT / 'gte_reranker_pair_scores.csv').open(encoding='utf-8', newline='')))
stored_ceiling = list(csv.DictReader((OUTPUT_ROOT / 'gte_reranker_candidate_ceiling.csv').open(encoding='utf-8', newline='')))
assert (len(stored_per_query), len(stored_summary), len(stored_pairs), len(stored_ceiling)) == (180, 36, 1857, 120)
assert len({(row['query_id'], row['chunk_id']) for row in stored_pairs}) == 1857
assert all(math.isfinite(float(row['reranker_logit'])) for row in stored_pairs)
assert all(0 <= float(row['document_truncation_ratio']) <= 1 and int(row['input_tokens']) <= MAX_LENGTH for row in stored_pairs)
assert all(int(row['candidate_count']) == int(row['top_k']) for row in stored_ceiling)
assert tree_snapshot(MODEL_ROOT) == cache_before['model'] and tree_snapshot(CUSTOM_HUB_ROOT) == cache_before['custom_hub'] and tree_snapshot(CUSTOM_MODULE_ROOT) == cache_before['custom_modules']
assert {name: sha256_file(path) for name, path in SOURCE_FILES.items()} == source_hashes_before
stored_json = json.loads((OUTPUT_ROOT / 'gte_reranker_summary.json').read_text(encoding='utf-8'))
assert stored_json['integrity']['unique_pair_scored_once'] and stored_json['integrity']['source_and_cache_unchanged']
assert stored_json['execution']['network_calls'] == stored_json['execution']['api_calls'] == stored_json['execution']['new_embeddings'] == stored_json['execution']['package_installs'] == 0
print({'validation': 'PASS', 'rows': [180, 36, 1857, 120], 'baseline_exact': True, 'top20_prefix': True, 'finite_range_permutation_no_leak': True, 'source_model_custom_cache_unchanged': True})


{'validation': 'PASS', 'rows': [180, 36, 1857, 120], 'baseline_exact': True, 'top20_prefix': True, 'finite_range_permutation_no_leak': True, 'source_model_custom_cache_unchanged': True}
